# Cat Skin Disease: Inference and Training

This notebook focuses on the skin disease model, including dataset exploration, inference using the existing `.keras` model, and exporting the model to a pickle format as requested.

In [2]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import pickle

## 1. Load Dataset and Verify Structure

In [3]:
DATASET_PATH = "../datasets/CAT_SKIN_DISEASE"
CLASSES = ['Flea_Allergy', 'Health', 'Ringworm', 'Scabies']

for cls in CLASSES:
    cls_path = os.path.join(DATASET_PATH, cls)
    if os.path.exists(cls_path):
        n_images = len(os.listdir(cls_path))
        print(f"Category: {cls} - Total Images: {n_images}")
    else:
        print(f"Category: {cls} NOT FOUND!")

Category: Flea_Allergy - Total Images: 250
Category: Health - Total Images: 250
Category: Ringworm - Total Images: 250
Category: Scabies - Total Images: 250


## 2. Load Existing .keras Model

In [4]:
MODEL_PATH = "../models/weights/cat_disease_model.keras"
if os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    model.summary()
    print("Model loaded successfully!")
else:
    print("Model NOT FOUND at weights/cat_disease_model.keras")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,508,430 (127.82 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 22,338,954 (85.22 MB)

Model loaded successfully!


## 3. Preprocessing and Inference Example

In [5]:
def predict_disease(img_path, model):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img_norm = img.astype('float32') / 255.0
    
    pred = model.predict(np.expand_dims(img_norm, 0), verbose=0)
    class_idx = np.argmax(pred)
    return CLASSES[class_idx], pred[0][class_idx]

# Test with a sample image if folder not empty
flea_path = os.path.join(DATASET_PATH, "Flea_Allergy")
if os.path.exists(flea_path) and len(os.listdir(flea_path)) > 0:
    sample_img = os.path.join(flea_path, os.listdir(flea_path)[0])
    label, conf = predict_disease(sample_img, model)
    print(f"Sample Inference - Image: {os.path.basename(sample_img)} -> Prediction: {label} ({conf*100:.2f}%)")

Sample Inference - Image: thumb (39).jpg -> Prediction: Flea_Allergy (99.97%)


## 4. Convert Keras Model to Pickle Format
Since Keras models are not easily picklable in their entirety (including architecture), we will pickle the model's weights and architecture separately or use a wrapper.

In [6]:
PKL_PATH = "../models/weights/cat_disease_model.pkl"

try:
    # Standard practice: Use pickle for non-Keras models, but for the user requested, 
    # we will use the keras functional approach to pickle weights and config.
    model_data = {
        'weights': model.get_weights(),
        'config': model.get_config(),
        'class_names': CLASSES
    }
    
    with open(PKL_PATH, 'wb') as f:
        pickle.dump(model_data, f)
    
    print(f"Successfully converted and saved model to {PKL_PATH}")
except Exception as e:
    print(f"Feailed to save pickle: {e}")

Successfully converted and saved model to ../models/weights/cat_disease_model.pkl
